In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import requests 
import io 
from sklearn.linear_model import LinearRegression


In [ ]:
ruta = r"C:\Users\jaime\Desktop\Jaime_LLorca\proyectos\prediccion-electrica\data\raw\mibgas\MIBGAS_Data_2023.csv"
with open(ruta, encoding="utf-8") as f:
    texto = f.read()
print(texto[:300])

In [ ]:
def parsear(texto):
    df = pd.read_csv(io.StringIO(texto), sep=";", skiprows=1)
    df = df[df["Product"] == "GDAES_D+1"]

    # El nombre de la columna de precio lleva la unidad, se busca por contenido
    col_precio = [c for c in df.columns if "Price" in c][0]
    df = df[["First Day Delivery", col_precio]].rename(
        columns={"First Day Delivery": "fecha_entrega", col_precio: "precio_gas"}
    )

    df["fecha_entrega"] = pd.to_datetime(df["fecha_entrega"], format="%d/%m/%Y")
    df=df.sort_values("fecha_entrega").reset_index(drop=True)
    return df.dropna(subset=["precio_gas"]).reset_index(drop=True)

df = parsear(texto)
df.head()

In [ ]:
def descargar_anio(anio):
    url = f"https://www.mibgas.es/es/file-access/MIBGAS_Data_{anio}.csv?path=AGNO_{anio}/XLS"
    r = requests.get(url, timeout=30)
    r.raise_for_status()

    # El fichero es UTF-8; sin esto la "ó" de "Emisión" puede salir rota
    r.encoding = "utf-8"
    return r.text

df_web = parsear(descargar_anio(2023))
df_web.equals(df)

In [ ]:
anios = [2023, 2024, 2025, 2026]
lista = []
for anio in anios:
    df_anio = parsear(descargar_anio(anio))
    print(anio, len(df_anio), df_anio["fecha_entrega"].min().date(), df_anio["fecha_entrega"].max().date())
    lista.append(df_anio)

df_gas = pd.concat(lista, ignore_index=True)

In [ ]:

print("Duplicados:", df_gas["fecha_entrega"].duplicated().sum())

dups = df_gas[df_gas["fecha_entrega"].duplicated(keep=False)]
precios_distintos = dups.groupby("fecha_entrega")["precio_gas"].nunique().gt(1).sum()
print("Fechas duplicadas con precio distinto:", precios_distintos)

In [ ]:
df_gas = (
    df_gas.drop_duplicates(subset="fecha_entrega", keep="last")
    .sort_values("fecha_entrega")
    .reset_index(drop=True)
)

inicio, fin = df_gas["fecha_entrega"].min(), df_gas["fecha_entrega"].max()
print("Rango de fechas:", inicio.date(), "a", fin.date())

todas = pd.date_range(inicio, fin, freq="D")
huecos = todas.difference(df_gas["fecha_entrega"])
print(len(df_gas), "días de", len(todas), "posibles. Huecos:", len(huecos))
huecos

In [ ]:
tf = pd.read_parquet("../data/processed/tabla_features.parquet", columns=["datetime_utc", "precio_espana"])

# Media diaria en hora de Madrid, no en UTC
luz = tf.set_index("datetime_utc")["precio_espana"].tz_convert("Europe/Madrid").resample("D").mean()
luz.index = luz.index.tz_localize(None)

# 2023-05-28, 2025-10-30 y 2025-11-27 no tienen ningún precio horario en la tabla
df_diario = df_gas.set_index("fecha_entrega").join(luz.rename("precio_luz"), how="inner")
print(df_diario.shape, "| días sin precio de luz:", df_diario["precio_luz"].isna().sum())
df_diario.tail()

In [ ]:


datos = df_diario.loc["2023":]

fig, ax_luz = plt.subplots(figsize=(13, 5))
ax_gas = ax_luz.twinx()

ax_luz.plot(datos.index, datos["precio_luz"], color="tab:blue", lw=0.8, label="Luz (media diaria)")
ax_gas.plot(datos.index, datos["precio_gas"], color="tab:orange", lw=1.2, label="Gas MIBGAS D+1")

ax_luz.set_ylabel("Luz [EUR/MWh]", color="tab:blue")
ax_gas.set_ylabel("Gas [EUR/MWh]", color="tab:orange")
ax_luz.tick_params(axis="y", colors="tab:blue")
ax_gas.tick_params(axis="y", colors="tab:orange")
ax_luz.set_title("Precio de la luz vs precio del gas")

lineas = ax_luz.get_lines() + ax_gas.get_lines()
ax_luz.legend(lineas, [l.get_label() for l in lineas], loc="upper center")
plt.tight_layout()
plt.show()

In [ ]:
corr_nivel = df_diario["precio_luz"].corr(df_diario["precio_gas"])

variacion = df_diario.pct_change(7, fill_method=None)
corr_variacion = variacion["precio_luz"].corr(variacion["precio_gas"])

print(f"Correlación en nivel:              {corr_nivel:.2f}")
print(f"Correlación de variaciones (7 días): {corr_variacion:.2f}")

In [ ]:
# La luz tiene días con media casi 0 EUR/MWh, y la variación porcentual se dispara
# (hasta +14.000 %): unos pocos días dominan Pearson. Dos alternativas robustas:
corr_spearman = variacion["precio_luz"].corr(variacion["precio_gas"], method="spearman")

diferencia = df_diario.diff(7)
corr_diferencia = diferencia["precio_luz"].corr(diferencia["precio_gas"])

print("Variación luz (7 días), máx.:", f"{variacion['precio_luz'].max():.0%}")
print(f"Spearman de variaciones:     {corr_spearman:.2f}")
print(f"Pearson de diferencias (EUR): {corr_diferencia:.2f}")

In [ ]:
# El gas entregado el día D se conoce al cerrar D-1: para predecir la luz
# de fecha_objetivo se usa el gas con entrega el día anterior
gas_d1 = df_gas[["fecha_entrega", "precio_gas"]].copy()
gas_d1["fecha_objetivo"] = gas_d1["fecha_entrega"] + pd.Timedelta(days=1)
gas_d1 = gas_d1[["fecha_objetivo", "precio_gas"]].rename(columns={"precio_gas": "gas_d1"})
gas_d1.tail()

In [ ]:
# Fecha en hora de Madrid, sin zona horaria (como fecha_objetivo de gas_d1) y a las 00:00,
# para que las horas del mismo día compartan fecha
tf["fecha_madrid"] = (
    tf["datetime_utc"]
    .dt.tz_convert("Europe/Madrid")
    .dt.tz_localize(None)
    .dt.normalize()
)
tf[["datetime_utc", "fecha_madrid"]].head(3)

In [ ]:
n_antes = len(tf)
tf = tf.merge(gas_d1, left_on="fecha_madrid", right_on="fecha_objetivo", how="left").drop(columns="fecha_objetivo")

# how="left": no se puede perder ninguna fila horaria
assert len(tf) == n_antes, f"{n_antes} filas antes y {len(tf)} después"
print("Filas:", len(tf))

In [ ]:
# Solo debería faltar el 01-01-2023: necesitaría el gas con entrega el 31-12-2022
print("Nulos en gas_d1:", tf["gas_d1"].isna().sum())
print("Fechas sin gas:", tf.loc[tf["gas_d1"].isna(), "fecha_madrid"].dt.date.unique())

# Auditoría a mano: gas_d1 del día D = precio de entrega del día D-1 en df_gas.
# 07-11-2026 es mañana: tiene que usar el gas con entrega 06-11, que ya está publicado
precio_por_entrega = df_gas.set_index("fecha_entrega")["precio_gas"]
for dia in ["2024-03-15", "2025-11-01", "2026-11-05", "2026-11-07"]:
    dia = pd.Timestamp(dia)
    valores = tf.loc[tf["fecha_madrid"] == dia, "gas_d1"].unique()
    esperado = precio_por_entrega[dia - pd.Timedelta(days=1)]
    estado = "OK" if len(valores) == 1 and valores[0] == esperado else "FALLO"
    print(f"{dia.date()}: gas_d1={valores} | entrega {(dia - pd.Timedelta(days=1)).date()}={esperado} -> {estado}")

In [ ]:
import sys
sys.path.append("..")

from xgboost import XGBRegressor
from src.evaluacion.backtesting import walk_forward
from src.features.feature_sets import get_feature_sets, TARGET

# Mismo esquema que el notebook 06: tabla completa, solo filas entrenables
df = pd.read_parquet("../data/processed/tabla_features.parquet").sort_values("datetime_utc")
df = df[df["entrenable"]]

# Las features oficiales se sacan antes de añadir columnas nuevas
feats = get_feature_sets(df)["predictivo"]
assert len(feats) == 97, f"se esperaban 97 features y hay {len(feats)}"
feats_gas = feats + ["gas_d1"]

df["fecha_madrid"] = df["datetime_utc"].dt.tz_convert("Europe/Madrid").dt.tz_localize(None).dt.normalize()
n_antes = len(df)
df = df.merge(gas_d1, left_on="fecha_madrid", right_on="fecha_objetivo", how="left").drop(columns="fecha_objetivo")
assert len(df) == n_antes

df["meses"] = df["datetime_utc"].dt.to_period("M")
# Solo meses completos: el mes en curso tiene pocos días y no es comparable con el notebook 06
df = df[df["meses"] < pd.Timestamp.now(tz="Europe/Madrid").tz_localize(None).to_period("M")]
print(f"{len(df)} filas | {df['meses'].nunique()} meses | nulos en gas_d1: {df['gas_d1'].isna().sum()}")

In [ ]:
def xgb_produccion():
    # Receta de producción (src/orquestacion/reentrenamiento.py)
    return XGBRegressor(
        n_estimators=400, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        tree_method="hist", random_state=42, n_jobs=-1,
    )

# Listas donde cada función guarda sus predicciones, mes a mes
preds_base, preds_gas = [], []

def predecir_base(tr, te):
    p = xgb_produccion().fit(tr[feats], tr[TARGET]).predict(te[feats])
    preds_base.append(pd.Series(p, index=te.index))
    return p

def predecir_gas(tr, te):
    p = xgb_produccion().fit(tr[feats_gas], tr[TARGET]).predict(te[feats_gas])
    preds_gas.append(pd.Series(p, index=te.index))
    return p

df_wf_base = walk_forward(df, TARGET, predecir_base)
df_wf_gas = walk_forward(df, TARGET, predecir_gas)

print(f"XGBoost             → MAE {df_wf_base['MAE'].mean():.2f} ± {df_wf_base['MAE'].std():.2f}")
print(f"XGBoost + gas_d1    → MAE {df_wf_gas['MAE'].mean():.2f} ± {df_wf_gas['MAE'].std():.2f}")

## Walk-forward: con y sin gas

In [ ]:
comparacion = pd.DataFrame({
    "mes": df_wf_base["mes"],
    "MAE_base": df_wf_base["MAE"],
    "MAE_gas": df_wf_gas["MAE"],
})
comparacion["diferencia"] = comparacion["MAE_gas"] - comparacion["MAE_base"]

resumen = pd.DataFrame({
    "MAE": [df_wf_base["MAE"].mean(), df_wf_gas["MAE"].mean()],
    "desviación": [df_wf_base["MAE"].std(), df_wf_gas["MAE"].std()],
    "meses que gana": [
        (comparacion["diferencia"] > 0).sum(),
        (comparacion["diferencia"] < 0).sum(),
    ],
}, index=["XGBoost", "XGBoost + gas_d1"]).round(2)

print(f"{len(comparacion)} meses evaluados")
display(resumen)

In [ ]:
# Agosto-octubre de 2026: es cuando sube el gas, donde más debería notarse
ultimos = comparacion[comparacion["mes"].between(pd.Period("2026-08", "M"), pd.Period("2026-10", "M"))]
display(ultimos.round(2))
print(f"MAE medio ago-oct 2026: base {ultimos['MAE_base'].mean():.2f} | con gas {ultimos['MAE_gas'].mean():.2f}")

## Sesgo: ¿corrige el gas el error de nivel?

In [ ]:
# Sesgo = predicción - real (negativo: el modelo predice por debajo del precio real)
pred_base = pd.concat(preds_base)
pred_gas = pd.concat(preds_gas)
real = df.loc[pred_base.index, TARGET]

sesgo = pd.DataFrame({
    "mes": df.loc[pred_base.index, "meses"],
    "sesgo_base": pred_base - real,
    "sesgo_gas": pred_gas.loc[pred_base.index] - real,
}).groupby("mes").mean().round(2)

# Media del valor absoluto: los sesgos de meses distintos se compensan si se promedian con signo
print(f"|sesgo| medio por mes, todo el periodo: base {sesgo['sesgo_base'].abs().mean():.2f} | "
      f"con gas {sesgo['sesgo_gas'].abs().mean():.2f}")
display(sesgo[sesgo.index.year == 2026])

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
x = sesgo.index.to_timestamp()
ax.plot(x, sesgo["sesgo_base"], marker="o", label="XGBoost")
ax.plot(x, sesgo["sesgo_gas"], marker="o", label="XGBoost + gas_d1")
ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("Sesgo medio (€/MWh)")
ax.set_title("Sesgo mensual en el walk-forward (predicción − real)")
ax.legend()
plt.tight_layout()
plt.show()

## Modelo híbrido: regresión lineal sobre el gas + XGBoost sobre el residuo

El árbol no extrapola: con el gas a 90–100 €/MWh (fuera del rango de entrenamiento) actúa como si valiera el máximo visto. La idea es separar el **nivel** del precio, que sigue al gas y que una recta sí puede extrapolar, de la **forma** (horas, eólica, demanda...), que sigue aprendiendo XGBoost.

1. `LinearRegression` de `precio_espana` sobre `gas_d1`, ajustada solo con el tramo de entrenamiento de cada mes.
2. XGBoost (receta de producción) sobre el residuo `precio_espana − recta(gas_d1)`.
3. Predicción = recta(gas_d1) + XGBoost(features).

Dos variantes: el XGBoost del residuo con las 97 features oficiales, y con las 97 + `gas_d1`.

In [ ]:
from sklearn.linear_model import LinearRegression

preds_hib, preds_hib_gas = [], []
rectas = []  # ordenada y pendiente de cada mes, para ver cómo evoluciona la recta

def ajustar_hibrido(tr, te, feats_xgb):
    recta = LinearRegression().fit(tr[["gas_d1"]], tr[TARGET])
    residuo = tr[TARGET] - recta.predict(tr[["gas_d1"]])
    xgb = xgb_produccion().fit(tr[feats_xgb], residuo)
    return recta.predict(te[["gas_d1"]]) + xgb.predict(te[feats_xgb]), recta

def predecir_hibrido(tr, te):
    p, recta = ajustar_hibrido(tr, te, feats)
    rectas.append({
        "mes": te["meses"].iloc[0], "ordenada": recta.intercept_, "pendiente": recta.coef_[0],
        "gas_max_train": tr["gas_d1"].max(), "gas_medio_test": te["gas_d1"].mean(),
    })
    preds_hib.append(pd.Series(p, index=te.index))
    return p

def predecir_hibrido_gas(tr, te):
    p, _ = ajustar_hibrido(tr, te, feats_gas)
    preds_hib_gas.append(pd.Series(p, index=te.index))
    return p

df_wf_hib = walk_forward(df, TARGET, predecir_hibrido)
df_wf_hib_gas = walk_forward(df, TARGET, predecir_hibrido_gas)
# Regla del proyecto: todo se compara con el naive D-1
df_wf_naive = walk_forward(df, TARGET, lambda tr, te: te["precio_lag_24"].to_numpy())

In [ ]:
# Pendiente = €/MWh de luz por cada €/MWh de gas. En 2026 el gas del test supera el máximo
# visto en entrenamiento: ahí es donde la recta tiene que aportar lo que el árbol no puede
rectas_df = pd.DataFrame(rectas).set_index("mes").round(2)
display(rectas_df.loc[rectas_df.index.year == 2026])

### Comparación: naive, XGBoost, XGBoost + gas y los dos híbridos

In [ ]:
modelos_wf = {
    "Naive D-1": df_wf_naive,
    "XGBoost": df_wf_base,
    "XGBoost + gas_d1": df_wf_gas,
    "Híbrido": df_wf_hib,
    "Híbrido + gas_d1": df_wf_hib_gas,
}
predicciones = {
    "Naive D-1": df.loc[real.index, "precio_lag_24"],
    "XGBoost": pred_base,
    "XGBoost + gas_d1": pred_gas,
    "Híbrido": pd.concat(preds_hib),
    "Híbrido + gas_d1": pd.concat(preds_hib_gas),
}
meses_real = df.loc[real.index, "meses"]
sesgos = pd.DataFrame({n: p.loc[real.index] - real for n, p in predicciones.items()}).groupby(meses_real).mean()
maes = pd.DataFrame({n: w.set_index("mes")["MAE"] for n, w in modelos_wf.items()})

resumen_hib = pd.DataFrame({
    "MAE": maes.mean(),
    "desviación": maes.std(),
    "meses que gana a XGBoost": maes.lt(maes["XGBoost"], axis=0).sum(),
    "|sesgo| medio": sesgos.abs().mean(),
    "MAE sep-oct 2026": maes.loc[pd.Period("2026-09", "M"):pd.Period("2026-10", "M")].mean(),
    "MAE mar-may 2026": maes.loc[pd.Period("2026-03", "M"):pd.Period("2026-05", "M")].mean(),
}).round(2)
resumen_hib.loc["XGBoost", "meses que gana a XGBoost"] = None
print(f"{len(maes)} meses evaluados")
display(resumen_hib)

In [ ]:
# 2026 mes a mes: el otoño (gas alto) y la primavera (donde el gas empeoraba) son la prueba
print("MAE por mes")
display(maes.loc[maes.index.year == 2026].round(2))
print("Sesgo por mes (predicción − real)")
display(sesgos.loc[sesgos.index.year == 2026].round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
x = sesgos.index.to_timestamp()
for nombre in ["XGBoost", "XGBoost + gas_d1", "Híbrido", "Híbrido + gas_d1"]:
    ax.plot(x, sesgos[nombre], marker="o", ms=3, label=nombre)
ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("Sesgo medio (€/MWh)")
ax.set_title("Sesgo mensual en el walk-forward: XGBoost vs híbrido")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Ensemble sencillo: media de XGBoost + gas_d1 e Híbrido + gas_d1.
# En otoño sus sesgos tienen signo contrario (uno predice por debajo y otro por encima),
# así que al promediarlos deberían compensarse en parte
nombre_ens = "Ensemble (XGB + gas | Híbrido + gas)"
pred_ens = (predicciones["XGBoost + gas_d1"].loc[real.index]
            + predicciones["Híbrido + gas_d1"].loc[real.index]) / 2
predicciones[nombre_ens] = pred_ens

# MAE y sesgo por mes, igual que para el resto de modelos (MAE mensual = media del error absoluto del mes)
maes[nombre_ens] = (pred_ens - real).abs().groupby(meses_real).mean()
sesgos[nombre_ens] = (pred_ens - real).groupby(meses_real).mean()

resumen_ens = pd.DataFrame({
    "MAE": maes.mean(),
    "desviación": maes.std(),
    "meses que gana a XGBoost": maes.lt(maes["XGBoost"], axis=0).sum(),
    "|sesgo| medio": sesgos.abs().mean(),
    "MAE sep-oct 2026": maes.loc[pd.Period("2026-09", "M"):pd.Period("2026-10", "M")].mean(),
    "MAE mar-may 2026": maes.loc[pd.Period("2026-03", "M"):pd.Period("2026-05", "M")].mean(),
}).round(2)
resumen_ens.loc["XGBoost", "meses que gana a XGBoost"] = None
display(resumen_ens)

print("2026 mes a mes: MAE y sesgo")
cols = ["XGBoost", "XGBoost + gas_d1", "Híbrido + gas_d1", nombre_ens]
display(pd.concat({
    "MAE": maes.loc[maes.index.year == 2026, cols],
    "sesgo": sesgos.loc[sesgos.index.year == 2026, cols],
}, axis=1).round(2))

### Ensemble con peso dinámico

Predicción = `w · Híbrido + gas_d1 + (1 − w) · XGBoost + gas_d1`, con `w` en una rejilla de 0 a 1 en pasos de 0,1.

- **Peso dinámico (honesto):** para cada mes se elige el `w` que minimiza el MAE medio de los meses **anteriores** y se aplica al mes actual. Esos meses anteriores ya son predicciones fuera de muestra del walk-forward, así que no se usa nada del mes que se evalúa. El primer mes no tiene historia y usa 0,5.
- **Curva MAE frente a `w` en todo el periodo:** solo es un diagnóstico. Elegir el `w` con esta curva sería mirar el futuro, así que su mínimo es optimista.

In [ ]:
import numpy as np

p_xgb_gas = predicciones["XGBoost + gas_d1"].loc[real.index]
p_hib_gas = predicciones["Híbrido + gas_d1"].loc[real.index]
rejilla_w = np.round(np.arange(0, 1.01, 0.1), 1)

# MAE de cada mes para cada w de la rejilla: filas = meses, columnas = w.
# Es la base de las dos cosas: el peso dinámico y la curva de diagnóstico
mae_por_w = pd.DataFrame({
    w: (w * p_hib_gas + (1 - w) * p_xgb_gas - real).abs().groupby(meses_real).mean()
    for w in rejilla_w
})

# Peso dinámico: para el mes i, el w con menor MAE medio en los meses 0..i-1 (solo pasado).
# El primer mes no tiene historia: 0,5. Si hay empate, idxmin se queda con el w más bajo
pesos = {}
for i, mes in enumerate(mae_por_w.index):
    pesos[mes] = 0.5 if i == 0 else mae_por_w.iloc[:i].mean().idxmin()
pesos = pd.Series(pesos, name="w")

# Cada hora usa el peso elegido para su mes
w_hora = meses_real.map(pesos)
pred_dinamico = w_hora * p_hib_gas + (1 - w_hora) * p_xgb_gas

nombre_din = "Ensemble peso dinámico"
predicciones[nombre_din] = pred_dinamico
maes[nombre_din] = (pred_dinamico - real).abs().groupby(meses_real).mean()
sesgos[nombre_din] = (pred_dinamico - real).groupby(meses_real).mean()

cols = ["XGBoost", "XGBoost + gas_d1", "Híbrido + gas_d1", nombre_ens, nombre_din]
resumen_din = pd.DataFrame({
    "MAE": maes[cols].mean(),
    "|sesgo| medio": sesgos[cols].abs().mean(),
    "MAE sep-oct 2026": maes.loc[pd.Period("2026-09", "M"):pd.Period("2026-10", "M"), cols].mean(),
    "MAE mar-may 2026": maes.loc[pd.Period("2026-03", "M"):pd.Period("2026-05", "M"), cols].mean(),
}).round(2)
display(resumen_din)

# Serie de pesos: w = 1 es todo Híbrido + gas, w = 0 es todo XGBoost + gas
print("Peso elegido cada mes (w = peso del Híbrido + gas_d1):")
display(pesos.to_frame().T)

fig, ax = plt.subplots(figsize=(13, 3))
ax.step(pesos.index.to_timestamp(), pesos.values, where="post", marker="o", ms=3)
ax.axhline(0.5, color="grey", ls="--", lw=0.8, label="w = 0,5 (media simple)")
ax.set_ylim(-0.05, 1.05)
ax.set_ylabel("w (peso del híbrido)")
ax.set_title("Peso dinámico elegido cada mes con los meses anteriores")
ax.legend()
plt.tight_layout()
# plt.show()

In [ ]:
# DIAGNÓSTICO, no evaluación: MAE medio de todo el periodo para cada w fijo.
# Para elegir w aquí haría falta conocer todos los meses de antemano, así que su mínimo es optimista
curva = mae_por_w.mean()
w_opt = curva.idxmin()
print(f"MAE con w = 0,5: {curva[0.5]:.2f} | mínimo (optimista): {curva.min():.2f} con w = {w_opt} | "
      f"peso dinámico (honesto): {maes[nombre_din].mean():.2f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(curva.index, curva.values, marker="o")
ax.axvline(0.5, color="grey", ls="--", lw=0.8, label="w = 0,5")
ax.axvline(w_opt, color="tab:red", ls=":", lw=1, label=f"mínimo (w = {w_opt})")
ax.axhline(maes[nombre_din].mean(), color="tab:green", lw=0.8, label="peso dinámico (honesto)")
ax.set_xlabel("w (0 = solo XGBoost + gas_d1, 1 = solo Híbrido + gas_d1)")
ax.set_ylabel("MAE medio mensual (€/MWh)")
ax.set_title("MAE frente a w en todo el periodo (diagnóstico, optimista)")
ax.legend()
plt.tight_layout()
plt.show()